## 1. RESTful URL 설계 원칙

### 기본 규칙

| 나쁜 예 | 좋은 예 | 이유 |
|---|---|---|
| `/getUsers` | `GET /users` | URL엔 동사 대신 명사(자원)만, 행위는 HTTP 메서드로 |
| `/user/delete/5` | `DELETE /users/5` | 메서드가 행위를 표현 |
| `/users/5/getPosts` | `GET /users/5/posts` | 계층 관계는 경로로 표현 |

### 지금 만드신 인증 API에 적용

```python
# 나쁜 예 - 동사가 URL에 들어감
@app.post("/sendVerificationCode")
@app.post("/checkVerificationCode")

# 좋은 예 - 자원 중심
@app.post("/api/v1/auth/verification-codes")       # 인증번호 생성(발송)
@app.post("/api/v1/auth/verification-codes/verify") # 인증번호 확인
```

---

## 2. HTTP 메서드와 상태 코드 제대로 쓰기

### 메서드 구분

```python
from fastapi import FastAPI, status

app = FastAPI()

@app.get("/api/v1/users/{user_id}")        # 조회
async def get_user(user_id: int): ...

@app.post("/api/v1/users", status_code=status.HTTP_201_CREATED)  # 생성
async def create_user(): ...

@app.patch("/api/v1/users/{user_id}")      # 부분 수정
async def update_user(user_id: int): ...

@app.delete("/api/v1/users/{user_id}", status_code=status.HTTP_204_NO_CONTENT)  # 삭제
async def delete_user(user_id: int): ...
```

### 상태 코드 치트시트 (실무에서 자주 쓰는 것만)

| 코드 | 의미 | 언제 쓰나 |
|---|---|---|
| 200 | OK | 조회/수정 성공 |
| 201 | Created | 생성 성공 (회원가입 성공 시) |
| 204 | No Content | 삭제 성공 (응답 바디 없음) |
| 400 | Bad Request | 입력값 자체가 잘못됨 (인증번호 형식 오류 등) |
| 401 | Unauthorized | 로그인 안 됨/토큰 없음 |
| 403 | Forbidden | 로그인은 했지만 권한 없음 |
| 404 | Not Found | 자원이 없음 |
| 409 | Conflict | 중복 (이메일 이미 존재) |
| 422 | Unprocessable Entity | FastAPI가 pydantic 검증 실패 시 자동 반환 |
| 500 | Internal Server Error | 서버 자체 에러 |

지금 회원가입에서 이메일 중복이면 **400이 아니라 409**를 쓰는 게 더 정확한 표현입니다.

---

## 3. 요청/응답 스키마 — Pydantic으로 검증 자동화

### 요청 검증

```python
from pydantic import BaseModel, EmailStr, field_validator

class SignupRequest(BaseModel):
    email: EmailStr  # 형식 자동 검증, 아니면 422 자동 반환
    password: str

    @field_validator("password")
    @classmethod
    def password_min_length(cls, v):
        if len(v) < 8:
            raise ValueError("비밀번호는 8자 이상이어야 합니다")
        return v

@app.post("/api/v1/auth/signup")
async def signup(body: SignupRequest):
    # 여기 들어온 시점엔 이미 이메일 형식/비밀번호 길이 검증 끝난 상태
    ...
```

이렇게 하면 "이메일 형식 체크", "비밀번호 길이 체크" 같은 코드를 직접 if문으로 짤 필요가 없어집니다. 지금 인증 로직에 수동 검증 코드가 있다면 이 방식으로 바꾸면 코드가 훨씬 깔끔해집니다.

---

## 4. 표준 응답 포맷 통일하기

지금까지는 엔드포인트마다 응답 구조가 다를 가능성이 높습니다. 아래처럼 **공통 응답 스키마**를 만들어두면 어디서 호출하든 일관된 구조가 나옵니다.

```python
from typing import Generic, TypeVar, Optional
from pydantic import BaseModel

T = TypeVar("T")

class ApiResponse(BaseModel, Generic[T]):
    success: bool
    data: Optional[T] = None
    message: Optional[str] = None

class ErrorDetail(BaseModel):
    code: str
    message: str

class ApiErrorResponse(BaseModel):
    success: bool = False
    error: ErrorDetail
```

```python
@app.post("/api/v1/auth/signup", response_model=ApiResponse[dict])
async def signup(body: SignupRequest):
    user = create_user(body.email, body.password)
    return ApiResponse(success=True, data={"user_id": user.id}, message="회원가입 완료")
```

---

## 5. 에러 처리 — 예외를 전역에서 일관되게 잡기

지금 "예외처리 다 해서 없앴다"고 하셨는데, 각 함수마다 try/except를 흩어놓기보다 **전역 예외 핸들러**로 모아두는 게 실무 패턴입니다.

```python
from fastapi import Request
from fastapi.responses import JSONResponse

class AppException(Exception):
    def __init__(self, code: str, message: str, status_code: int = 400):
        self.code = code
        self.message = message
        self.status_code = status_code

@app.exception_handler(AppException)
async def app_exception_handler(request: Request, exc: AppException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"success": False, "error": {"code": exc.code, "message": exc.message}}
    )
```

```python
# 실제 사용 예
@app.post("/api/v1/auth/signup")
async def signup(body: SignupRequest):
    if user_exists(body.email):
        raise AppException(
            code="EMAIL_ALREADY_EXISTS",
            message="이미 가입된 이메일입니다.",
            status_code=409
        )
    ...
```

장점: 비즈니스 로직 코드에서는 `raise AppException(...)`만 던지면 되고, "어떻게 JSON으로 응답할지"는 한 곳(전역 핸들러)에서만 관리됩니다. 나중에 응답 형식을 바꿔야 할 때도 한 곳만 고치면 됩니다.

---

## 6. API 버저닝

```python
from fastapi import APIRouter

v1_router = APIRouter(prefix="/api/v1")

@v1_router.post("/auth/signup")
async def signup_v1(): ...

app.include_router(v1_router)
```

지금 규모에선 당장 필요 없어 보여도, 나중에 로그인 방식을 바꾸는 등 기존 클라이언트를 깨뜨릴 변경이 생기면 `/api/v2/`를 새로 만들어서 점진적으로 이전할 수 있습니다.

---

## 7. Swagger 자동 문서화 활용

FastAPI는 별도 작업 없이 `/docs`에서 문서가 뜨지만, 설명을 추가하면 훨씬 유용해집니다.

```python
@app.post(
    "/api/v1/auth/verification-codes/verify",
    summary="이메일 인증번호 확인",
    description="회원가입 시 발송된 6자리 인증번호를 검증합니다. 5분 내 미인증 시 만료됩니다.",
    responses={
        400: {"description": "인증번호 불일치 또는 만료"},
        404: {"description": "해당 이메일로 발송된 인증번호 없음"},
    }
)
async def verify_code(body: VerifyCodeRequest):
    ...
```

면접에서 "API 문서화 어떻게 관리하셨어요?"라는 질문에 이 화면(`/docs`) 캡처만 보여줘도 좋은 답변이 됩니다.

---

## 8. 페이지네이션 (목록 조회 시 필수)

유저 목록, 로그 목록처럼 데이터가 많아질 API가 있다면 처음부터 페이지네이션을 넣는 습관을 들이세요.

```python
from fastapi import Query

@app.get("/api/v1/users")
async def list_users(page: int = Query(1, ge=1), size: int = Query(20, le=100)):
    offset = (page - 1) * size
    users = get_users(limit=size, offset=offset)
    total = count_users()
    return ApiResponse(data={
        "items": users,
        "page": page,
        "size": size,
        "total": total
    })
```

`size`에 `le=100` 같은 상한을 걸어두는 것도 중요합니다 — 안 걸면 누군가 `size=1000000`으로 요청해서 서버에 부담을 줄 수 있습니다.

---

## 지금 프로젝트에 적용할 체크리스트

1. 지금 있는 엔드포인트들의 응답 구조가 다 다른지 확인 → `ApiResponse` 공통 스키마로 통일
2. 회원가입 실패 케이스들(이메일 중복, 인증번호 틀림 등)에 적절한 상태 코드(400/409) 매칭돼 있는지 점검
3. 흩어진 try/except를 전역 exception handler로 모으기
4. `/docs` 페이지 열어서 지금 API들이 잘 문서화되고 있는지 확인, summary/description 채워넣기
